In [ ]:
%load_ext autoreload
%autoreload 2

import random
import sys
from pathlib import Path
from types import SimpleNamespace

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Continuous scripts use float32 modules.
torch.set_default_dtype(torch.float32)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def find_root(current_path, marker="setup.py"):
    current_path = Path(current_path).resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / marker).exists():
            return parent
    return current_path


PROJECT_ROOT = find_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
ASSETS_DIR = PROJECT_ROOT / "experiments" / "shared" / "assets"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rl_methods.sbeed.features import (
    ContinuousNeuralRhoParam,
    ContinuousNeuralValueParam,
    ContinuousStateActionMLPModule,
    ContinuousStateMLPValueModule,
    RFFGaussianPolicyParam,
)
from rl_methods.sbeed.solvers import ContinuousSBEED

print(DEVICE)

In [ ]:
def evaluate_continuous_policy(
    solver,
    env_id=None,
    env=None,
    episodes=5,
    deterministic=True,
    seed=SEED,
    render=False,
):
    close_env = False
    if env is None:
        env = gym.make(env_id, render_mode="human" if render else None)
        close_env = True

    returns = []

    for ep in range(episodes):
        reset_result = env.reset(seed=seed + ep)
        obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result
        done = False
        ep_return = 0.0
        steps = 0

        while not done:
            action = solver.sample_action(obs, deterministic=deterministic, clip=True)
            step_result = env.step(action)

            if len(step_result) == 5:
                obs, reward, terminated, truncated, info = step_result
                done = terminated or truncated
            else:
                obs, reward, done, info = step_result

            ep_return += float(reward)
            steps += 1

        returns.append(ep_return)

    if close_env:
        env.close()

    returns = np.asarray(returns, dtype=float)
    print(f"eval_returns={returns}")
    print(f"mean={returns.mean():.3f} std={returns.std():.3f} min={returns.min():.3f} max={returns.max():.3f}")

    return returns

In [ ]:
PENDULUM_ENV_ID = "Pendulum-v1"
pendulum_env = gym.make(PENDULUM_ENV_ID)

pendulum_obs_dim = int(np.prod(pendulum_env.observation_space.shape))
pendulum_action_dim = int(np.prod(pendulum_env.action_space.shape))

pendulum_cfg = SimpleNamespace(
    episodes=5,
    initial_random_steps=256,
    collect_per_episode=128,
    updates_per_episode=10,
    batch_size=64,
    rollout_length=1,
    max_buffer_size=5000,
    hidden_size=64,
    rff_features=100,
    nu=None,
    init_log_std=-0.5,
    gamma=0.99,
    lambda_entropy=0.01,
    eta=1.0,
    lr_value=1e-3,
    lr_rho=1e-3,
    lr_policy=1e-3,
    fisher_damping=1e-2,
    cg_iters=10,
    tau=100.0,
    seed=SEED,
    device=DEVICE,
    log_every=1,
)

print("obs_dim:", pendulum_obs_dim)
print("action_dim:", pendulum_action_dim)
print("action_low:", pendulum_env.action_space.low)
print("action_high:", pendulum_env.action_space.high)


In [ ]:
pendulum_value_param = ContinuousNeuralValueParam(
    ContinuousStateMLPValueModule(
        obs_dim=pendulum_obs_dim,
        hidden_sizes=(pendulum_cfg.hidden_size, pendulum_cfg.hidden_size),
        dtype=torch.float32,
    )
)

pendulum_rho_param = ContinuousNeuralRhoParam(
    ContinuousStateActionMLPModule(
        obs_dim=pendulum_obs_dim,
        action_dim=pendulum_action_dim,
        hidden_sizes=(pendulum_cfg.hidden_size, pendulum_cfg.hidden_size),
        output_dim=1,
        dtype=torch.float32,
    )
)

pendulum_policy_param = RFFGaussianPolicyParam(
    obs_dim=pendulum_obs_dim,
    action_dim=pendulum_action_dim,
    num_features=pendulum_cfg.rff_features,
    nu=pendulum_cfg.nu,
    init_log_std=pendulum_cfg.init_log_std,
    dtype=torch.float32,
    seed=pendulum_cfg.seed,
)


In [ ]:
pendulum_solver = ContinuousSBEED(
    obs_dim=pendulum_obs_dim,
    action_dim=pendulum_action_dim,
    gamma=pendulum_cfg.gamma,
    value_param=pendulum_value_param,
    rho_param=pendulum_rho_param,
    policy_param=pendulum_policy_param,
    lambda_entropy=pendulum_cfg.lambda_entropy,
    eta=pendulum_cfg.eta,
    lr_value=pendulum_cfg.lr_value,
    lr_rho=pendulum_cfg.lr_rho,
    lr_policy=pendulum_cfg.lr_policy,
    batch_size=pendulum_cfg.batch_size,
    rollout_length=pendulum_cfg.rollout_length,
    max_buffer_size=pendulum_cfg.max_buffer_size,
    fisher_damping=pendulum_cfg.fisher_damping,
    cg_iters=pendulum_cfg.cg_iters,
    tau=pendulum_cfg.tau,
    seed=pendulum_cfg.seed,
    device=pendulum_cfg.device,
)

pendulum_result = pendulum_solver.run_env(
    pendulum_env,
    episodes=pendulum_cfg.episodes,
    initial_random_steps=pendulum_cfg.initial_random_steps,
    collect_per_episode=pendulum_cfg.collect_per_episode,
    updates_per_episode=pendulum_cfg.updates_per_episode,
    log_every=pendulum_cfg.log_every,
)

pendulum_returns = pendulum_result["episode_returns"]
pendulum_avg_return = float(np.mean(pendulum_returns[-10:])) if pendulum_returns else float("nan")

print(f"buffer_size={pendulum_result['buffer_size']}")
print(f"recent_avg_return={pendulum_avg_return:.3f}")
print("last_stats:", pendulum_result["last_stats"])


In [ ]:
pendulum_eval_returns = evaluate_continuous_policy(
    pendulum_solver,
    env_id=PENDULUM_ENV_ID,
    episodes=5,
    deterministic=True,
    seed=SEED,
)

plt.figure(figsize=(7, 4))
plt.plot(pendulum_returns, marker="o", label="train completed episodes")
plt.axhline(pendulum_eval_returns.mean(), linestyle="--", label="eval mean")
plt.xlabel("episode")
plt.ylabel("return")
plt.title("Pendulum SBEED Returns")
plt.legend()
plt.grid(True)
plt.show()

sample_obs, _ = pendulum_env.reset(seed=SEED)
sample_action = pendulum_solver.sample_action(sample_obs, deterministic=True)
sample_value = pendulum_solver.value(sample_obs).detach().cpu().item()
sample_rho = pendulum_solver.rho(sample_obs, sample_action).detach().cpu().item()

print("sample_obs:", sample_obs)
print("deterministic_action:", sample_action)
print("value(sample_obs):", sample_value)
print("rho(sample_obs, action):", sample_rho)


In [ ]:
def make_swimmer_env():
    last_error = None
    for env_id in ("Swimmer-v5", "Swimmer-v4"):
        try:
            return gym.make(env_id), env_id
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        "Could not create Swimmer-v5 or Swimmer-v4. "
        "Install Gymnasium MuJoCo extras and verify MuJoCo is available."
    ) from last_error


swimmer_env, SWIMMER_ENV_ID = make_swimmer_env()

swimmer_obs_dim = int(np.prod(swimmer_env.observation_space.shape))
swimmer_action_dim = int(np.prod(swimmer_env.action_space.shape))

swimmer_cfg = SimpleNamespace(
    episodes=5,
    initial_random_steps=1024,
    collect_per_episode=512,
    updates_per_episode=25,
    batch_size=128,
    rollout_length=1,
    max_buffer_size=20000,
    hidden_size=128,
    rff_features=100,
    nu=None,
    init_log_std=-0.5,
    gamma=0.99,
    lambda_entropy=0.01,
    eta=1.0,
    lr_value=1e-3,
    lr_rho=1e-3,
    lr_policy=1e-3,
    fisher_damping=1e-2,
    cg_iters=10,
    tau=100.0,
    seed=SEED,
    device=DEVICE,
    log_every=1,
)

print("env_id:", SWIMMER_ENV_ID)
print("obs_dim:", swimmer_obs_dim)
print("action_dim:", swimmer_action_dim)
print("action_low:", swimmer_env.action_space.low)
print("action_high:", swimmer_env.action_space.high)


In [ ]:
swimmer_value_param = ContinuousNeuralValueParam(
    ContinuousStateMLPValueModule(
        obs_dim=swimmer_obs_dim,
        hidden_sizes=(swimmer_cfg.hidden_size, swimmer_cfg.hidden_size),
        dtype=torch.float32,
    )
)

swimmer_rho_param = ContinuousNeuralRhoParam(
    ContinuousStateActionMLPModule(
        obs_dim=swimmer_obs_dim,
        action_dim=swimmer_action_dim,
        hidden_sizes=(swimmer_cfg.hidden_size, swimmer_cfg.hidden_size),
        output_dim=1,
        dtype=torch.float32,
    )
)

swimmer_policy_param = RFFGaussianPolicyParam(
    obs_dim=swimmer_obs_dim,
    action_dim=swimmer_action_dim,
    num_features=swimmer_cfg.rff_features,
    nu=swimmer_cfg.nu,
    init_log_std=swimmer_cfg.init_log_std,
    dtype=torch.float32,
    seed=swimmer_cfg.seed,
)


In [ ]:
swimmer_solver = ContinuousSBEED(
    obs_dim=swimmer_obs_dim,
    action_dim=swimmer_action_dim,
    gamma=swimmer_cfg.gamma,
    value_param=swimmer_value_param,
    rho_param=swimmer_rho_param,
    policy_param=swimmer_policy_param,
    lambda_entropy=swimmer_cfg.lambda_entropy,
    eta=swimmer_cfg.eta,
    lr_value=swimmer_cfg.lr_value,
    lr_rho=swimmer_cfg.lr_rho,
    lr_policy=swimmer_cfg.lr_policy,
    batch_size=swimmer_cfg.batch_size,
    rollout_length=swimmer_cfg.rollout_length,
    max_buffer_size=swimmer_cfg.max_buffer_size,
    fisher_damping=swimmer_cfg.fisher_damping,
    cg_iters=swimmer_cfg.cg_iters,
    tau=swimmer_cfg.tau,
    seed=swimmer_cfg.seed,
    device=swimmer_cfg.device,
)

swimmer_result = swimmer_solver.run_env(
    swimmer_env,
    episodes=swimmer_cfg.episodes,
    initial_random_steps=swimmer_cfg.initial_random_steps,
    collect_per_episode=swimmer_cfg.collect_per_episode,
    updates_per_episode=swimmer_cfg.updates_per_episode,
    log_every=swimmer_cfg.log_every,
)

swimmer_returns = swimmer_result["episode_returns"]
swimmer_avg_return = float(np.mean(swimmer_returns[-10:])) if swimmer_returns else float("nan")

print(f"buffer_size={swimmer_result['buffer_size']}")
print(f"recent_avg_return={swimmer_avg_return:.3f}")
print("last_stats:", swimmer_result["last_stats"])


In [ ]:
swimmer_eval_returns = evaluate_continuous_policy(
    swimmer_solver,
    env_id=SWIMMER_ENV_ID,
    episodes=5,
    deterministic=True,
    seed=SEED,
)

plt.figure(figsize=(7, 4))
plt.plot(swimmer_returns, marker="o", label="train completed episodes")
plt.axhline(swimmer_eval_returns.mean(), linestyle="--", label="eval mean")
plt.xlabel("episode")
plt.ylabel("return")
plt.title(f"{SWIMMER_ENV_ID} SBEED Returns")
plt.legend()
plt.grid(True)
plt.show()

sample_obs, _ = swimmer_env.reset(seed=SEED)
sample_action = swimmer_solver.sample_action(sample_obs, deterministic=True)
sample_value = swimmer_solver.value(sample_obs).detach().cpu().item()
sample_rho = swimmer_solver.rho(sample_obs, sample_action).detach().cpu().item()

print("sample_obs:", sample_obs)
print("deterministic_action:", sample_action)
print("value(sample_obs):", sample_value)
print("rho(sample_obs, action):", sample_rho)
